# 14.最短経路: Shortest Path
有効グラフの辺に、重みなどの数値が設定されている場合、ネットワークと呼ぶ。
重みとして、距離、時間、コストなどを設定している場合、始点から各頂点への最短経路を求める問題を最短経路問題と呼ぶ。
最短経路問題を解くアルゴリズムとして、ダイクストラ法が有名である。

Directed graphs with weights on edges are called networks. When weights represent distance, time, cost, etc., the problem of finding the shortest path from a starting point to each vertex is called the shortest path problem. Dijkstra's algorithm is a well-known algorithm for solving the shortest path problem.

## 全ての辺の距離が同じ場合: Case of Unweighted Graphs
全ての辺の距離が同じ場合、最短経路問題は幅優先探索で解くことができる。
When all edges have the same weight, the shortest path problem can be solved using breadth-first search.

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import random
from collections import deque
from typing import NamedTuple,TypeVar, Generic, Hashable

In [ ]:
def bfs(graph:nx.DiGraph, start:Hashable)->set[tuple[Hashable,Hashable]]:
    visited:set[Hashable] = set()
    queue:deque[tuple[Hashable, Hashable]] = deque()
    queue.append((None,start))
    used_edges:set[tuple[Hashable,Hashable]] = set()    
    while queue:
        prev_node, node = queue.popleft()
        if node not in visited:
            visited.add(node)
            if prev_node is not None:
                used_edges.add((prev_node, node))
            neighbors = list(graph.neighbors(node))
            neighbors.sort()
            for neighbor in neighbors:
                if neighbor not in visited:
                    queue.append((node, neighbor))
    return used_edges


In [ ]:
def define_network()->nx.DiGraph:#全ての辺の長さが1
    G = nx.DiGraph() # 有向グラフの作成: Create a directed graph
    # 辺の追加: Add edges
    edges = [
        (1, 2), (1, 3), (2, 4),(2, 5), (2, 7), (3, 4),(3, 6),
        (4, 5), (5, 7), (6, 5), (7, 6)
    ]
    G.add_edges_from(edges)
    # 頂点の位置を設定: オプション
    # Set node positions (optional)
    positions = {
        1: (0, 0), 2: (1, 1), 3: (1, -1),
        4: (1, 0), 5: (2, 0), 6: (3, -1), 7: (3, 1)
    }
    nx.set_node_attributes(G, positions, 'pos')
    weight = {edge: 1 for edge in edges}  # 全ての辺の重みを1に設定: Set all edge weights to 1
    nx.set_edge_attributes(G, weight, 'weight')
    return G

In [ ]:
g = define_network()
used_edges = bfs(g, 1)
# グラフの描画: Draw the graph
pos = nx.get_node_attributes(g, 'pos')
nx.draw(g, pos, with_labels=True, node_color='lightblue', node_size=700)
nx.draw_networkx_edges(g, pos, edgelist=used_edges, edge_color='red', width=2)
plt.title("Shortest Path from Node 1 (Red Edges)")
plt.show()

In [ ]:
class Vertex(NamedTuple):
    """
    頂点、始点からの距離、前の頂点を表すタプル
    A tuple representing a vertex, its distance from the start, and the previous vertex.
    """
    id: Hashable
    p: float
    q:Hashable = None

    def __lt__(self, other):
        return self.p < other.p
    def __gt__(self, other):
        return self.p > other.p

# ヒープが格納する要素の型を定義: Define the type of elements stored in the heap
K = TypeVar('K')
class BinaryHeap(Generic[K]):
    """
    最小ヒープの簡単な実装（リストを使用）
    A simple implementation of a min-heap using a list.
    """
    def __init__(self):
        self._data:list[K|None] = [None]  # 1-based index; index 0 is unused
        self._size = 0
    
    def min_heapify(self, index):
        smallest = index
        left = 2 * index
        right = 2 * index + 1
        
        if left <= self._size and self._data[left] < self._data[smallest]:
            smallest = left
        if right <= self._size and self._data[right] < self._data[smallest]:
            smallest = right
        
        if smallest != index:
            self._data[index], self._data[smallest] = self._data[smallest], self._data[index]
            self.min_heapify(smallest)
    
    def build_min_heap(self, array:list[K]):
        self._data = [None] + array[:]  # 1-based index
        self._size = len(array)
        for i in range(self._size // 2, 0, -1):
            self.min_heapify(i)

    def insert(self, value: K):
        self._data.append(value)
        self._size += 1
        self._shift_up(self._size)
    def _shift_up(self, index):
        parent = index // 2
        if parent > 0 and self._data[index] < self._data[parent]:
            self._data[index], self._data[parent] = self._data[parent], self._data[index]
            self._shift_up(parent)
    
    def pop_min(self) -> K | None:
        if self.is_empty:
            raise IndexError("Heap is empty")
        min_value = self._data[1]
        self._data[1] = self._data[self._size]
        self._data.pop()  # Remove the last element
        self._size -= 1
        if not self.is_empty:
            self.min_heapify(1)
        return min_value
    @property
    def is_empty(self):
        return self._size == 0
    @property
    def heap_list(self) -> list[K | None]:
        return self._data[1:]  # Exclude the first None element

In [ ]:
def dijkstra(graph:nx.DiGraph, start:Hashable) -> set[Vertex]:
    """
    ダイクストラ法による最短経路探索
    Find the shortest path using Dijkstra's algorithm.
    """
    # 準備: Preparation
    U: BinaryHeap[Vertex] = BinaryHeap() # 探索済みの頂点: Set of explored vertices
    W: set[Vertex] = set() # 距離確定の頂点: Set of vertices with confirmed shortest distance
    V:dict[Hashable, Vertex] = {node:Vertex(id=node, p=float('inf')) for node in graph.nodes()} # 全ての頂点: All vertices
    # 初期値設定: Set initial values
    start_vertex = Vertex(id=start, p=0)
    V[start] = start_vertex
    U.insert(start_vertex)

    while not U.is_empty:
        # 最小距離の頂点を選ぶ: Select the vertex with the minimum distance
        w = U.pop_min()
        if w:
            ww = V[w.id]
            for v in graph.neighbors(w.id):
                if V[v].p > ww.p + graph[w.id][v]['weight']:
                    V[v] = Vertex(id=v, p=ww.p + graph[w.id][v]['weight'], q=w.id)
                    if V[v] in U.heap_list:
                        v_index = U.heap_list.index(V[v])  # Get the index of the vertex in the heap
                        U._data[v_index] = V[v]  # Update the vertex in the heap
                        U._shift_up(v_index)  # Restore the heap property
                    else:
                        U.insert(V[v])
            W.add(ww)
    return W
                    

In [ ]:
def define_network2()->nx.DiGraph:#全ての辺の長さが1ではない
    G = nx.DiGraph() # 有向グラフの作成: Create a directed graph
    # 辺の追加: Add edges
    edges = [
        (1, 2), (1, 3), (2, 4),(2, 5), (2, 7), (3, 4),(3, 6),
        (4, 5), (5, 7), (6, 5), (7, 6)
    ]
    G.add_edges_from(edges)
    # 頂点の位置を設定: オプション
    # Set node positions (optional)
    positions = {
        1: (0, 0), 2: (1, 1), 3: (1, -1),
        4: (1, 0), 5: (2, 0), 6: (3, -1), 7: (3, 1)
    }
    nx.set_node_attributes(G, positions, 'pos')
    weight = {edge: random.randint(1, 5) for edge in edges}  # 全ての辺の重みを1に設定: Set all edge weights to 1
    nx.set_edge_attributes(G, weight, 'weight')
    return G

In [ ]:
# g = define_network()
g = define_network2()
result = dijkstra(g, 1)
# for vertex in result:
#     print(f"Vertex {vertex.id}: Shortest distance from node 1 is {vertex.p}")
used_edges = [(vertex.q,vertex.id) for vertex in result if vertex.q is not None]
# print(used_edges)
# グラフの描画: Draw the graph
pos = nx.get_node_attributes(g, 'pos')
nx.draw(g, pos, with_labels=True, node_color='lightblue', node_size=700)
weight = nx.get_edge_attributes(g, 'weight')
nx.draw_networkx_edge_labels(g, pos, edge_labels=weight)
nx.draw_networkx_edges(g, pos, edgelist=used_edges, edge_color='red', width=2)
# 距離の表示
pos2 = {vertex.id: (pos[vertex.id][0]+0.2, pos[vertex.id][1]+0.1) for vertex in result}
nx.draw_networkx_labels(g, pos2, labels={vertex.id: vertex.p for vertex in result},font_color='red')
plt.title("Shortest Path from Node 1 (Red Edges)")
plt.show()